# Module 04: Search Chaining & the SLaM Pipeline

## Learning to Autolens

---

**Purpose:** Learn the strategy that makes complex lens modeling tractable: **search chaining**.
Instead of fitting all parameters at once (which is slow and error-prone for >15 parameters),
we break the problem into stages where each stage initializes the next. The **SLaM pipeline**
(Source, Light, and Mass) is PyAutoLens's standardized implementation of this strategy.

**Prerequisites:**
- Module 03 (Your First Lens Model) — model construction, non-linear search, results
- Understanding of why fitting many parameters simultaneously is difficult

**Key references:**
- Nightingale, Dye & Massey (2018), Sec. 6: *Automated Lens Modeling* (hereafter **Nightingale+18**)
- Nightingale et al. (2021): *PyAutoLens: Open-Source Strong Lensing*
- Speagle (2020): *dynesty* — for nested sampling details

---

## Table of Contents

1. [Why Search Chaining?](#1-why-search-chaining)
2. [The Mechanics: Prior Passing and Instance Fixing](#2-the-mechanics)
3. [A Simple Two-Search Chain](#3-a-simple-chain)
4. [The SLaM Pipeline: Architecture](#4-slam-architecture)
5. [SLaM Stage 1: SOURCE LP (Parametric Source)](#5-source-lp)
6. [SLaM Stage 2: SOURCE PIX (Pixelized Source)](#6-source-pix)
7. [SLaM Stage 3: LIGHT LP (Lens Light)](#7-light-lp)
8. [SLaM Stage 4: MASS TOTAL (Final Mass Model)](#8-mass-total)
9. [Putting It All Together: Full SLaM Run](#9-full-slam)
10. [Exercises](#10-exercises)

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import autolens as al
import autolens.plot as aplt
import autofit as af

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import the SLaM pipeline modules
import sys
sys.path.insert(0, str(Path("../../autolens_workspace_original").resolve()))
from slam import source_lp, source_pix, light_lp, mass_total

%matplotlib inline

print(f"PyAutoLens version: {al.__version__}")

---

## 1. Why Search Chaining? <a id="1-why-search-chaining"></a>

### The Curse of Dimensionality

A realistic lens model has **20–30+ free parameters**:

| Component | Profile | Parameters |
|-----------|---------|------------|
| Lens light | Sérsic (bulge + disk) | ~12 (centre, ell_comps, intensity, R_e, n × 2) |
| Lens mass | SIE + shear | 7 (centre, ell_comps, θ_E, γ₁, γ₂) |
| Source light | Sérsic | 7 (centre, ell_comps, intensity, R_e, n) |
| **Total** | | **~26** |

Nested sampling in 26 dimensions requires **millions** of likelihood evaluations.
Worse, the likelihood surface is highly **multimodal** — there are many local optima
where the source and lens light trade off against each other.

### The Solution: Divide and Conquer

Search chaining solves this by:
1. **Fit a simple model first** (few parameters, fast convergence)
2. **Use those results to initialize a more complex model** (prior passing)
3. **Fix some components while fitting others** (instance fixing)

Each search explores a smaller parameter space, guided by the results of the previous search.
The final search explores the full model but starts in the right region of parameter space.

### Analogy

It's like focusing a camera: first get roughly in focus (coarse adjustment), then
fine-tune. You don't start by trying all possible focus positions.

---

## 2. The Mechanics: Prior Passing and Instance Fixing <a id="2-the-mechanics"></a>

### Prior Passing: `result.model`

When Search 1 finds that $\theta_E = 1.62 \pm 0.05''$, we don't want Search 2 to
explore the full prior range $[0, 5]$ again. Instead, we **pass the posterior as a
prior** for Search 2:

```python
# Search 2 inherits θ_E ~ N(1.62, 0.05) from Search 1
mass = result_1.model.galaxies.lens.mass
```

PyAutoFit automatically converts posterior samples into Gaussian priors.

### Instance Fixing: `result.instance`

Sometimes we want to **completely fix** a component (zero free parameters):

```python
# Fix lens light to best-fit values from Search 1
bulge = result_1.instance.galaxies.lens.bulge
```

This is useful when:
- Fitting the source while the lens light is held fixed
- Fitting the mass while source and light are locked in

### Profile Evolution: `take_attributes()`

When upgrading from a simple to complex profile (e.g., SIS → SIE), shared
parameters can be transferred:

```python
# SIE inherits centre and θ_E priors from SIS
sie_mass = af.Model(al.mp.Isothermal)
sie_mass.take_attributes(result_sis.model.galaxies.lens.mass)
```

In [ ]:
# ============================================================
# LOAD EXAMPLE DATASET
# ============================================================
# We use the 'simple' dataset which includes lens light.
# This requires the full SLaM pipeline to model properly.
# ============================================================

dataset_name = "simple"
dataset_path = Path(
    "../../autolens_workspace_original/dataset/imaging/simple"
)

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.0,
)

dataset = dataset.apply_mask(mask=mask)

dataset_plotter = aplt.ImagingPlotter(dataset=dataset)
dataset_plotter.subplot_dataset()

---

## 3. A Simple Two-Search Chain <a id="3-a-simple-chain"></a>

Before diving into SLaM, let's build intuition with a simple two-search chain:

**Search 1**: Fit a simplified model (SIS, fixed centres, fixed Sérsic n).
~5 free parameters → fast convergence.

**Search 2**: Use Search 1 results as priors for the full model (SIE + shear,
free centres, free n). ~14 free parameters but starting near the truth.

In [ ]:
# ============================================================
# SEARCH 1: SIMPLIFIED MODEL (SIS, fixed centres)
# ============================================================
# Strategy: fit the absolute minimum to get a rough answer.
# Fix centres to (0,0), use spherical SIS (no ellipticity),
# fix Sérsic index. This reduces the problem to ~5 parameters.
# ============================================================

lens_1 = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=al.lp.Sersic,
    mass=al.mp.IsothermalSph,      # Spherical SIS (no ell_comps)
)

# Fix the lens centre to (0, 0) — reduces 2 free parameters
lens_1.mass.centre.centre_0 = 0.0
lens_1.mass.centre.centre_1 = 0.0
lens_1.bulge.centre.centre_0 = 0.0
lens_1.bulge.centre.centre_1 = 0.0

# Fix the Sérsic index to n=4 (de Vaucouleurs) — reduces 1 parameter
lens_1.bulge.sersic_index = 4.0

source_1 = af.Model(
    al.Galaxy,
    redshift=1.0,
    bulge=al.lp.SersicCore,
)
source_1.bulge.sersic_index = 1.0  # Fix to exponential

model_1 = af.Collection(
    galaxies=af.Collection(lens=lens_1, source=source_1)
)

print(f"Search 1: {model_1.total_free_parameters} free parameters")

search_1 = af.Nautilus(
    path_prefix=Path("output") / "module_04" / "chaining",
    name="search_1_simple",
    unique_tag=dataset_name,
    n_live=75,                     # Fewer live points for simple model
)

analysis = al.AnalysisImaging(dataset=dataset)
result_1 = search_1.fit(model=model_1, analysis=analysis)

print(f"\nSearch 1 complete. Best θ_E = {result_1.max_log_likelihood_instance.galaxies.lens.mass.einstein_radius:.3f}\"")

In [ ]:
# ============================================================
# SEARCH 2: FULL MODEL, INITIALIZED FROM SEARCH 1
# ============================================================
# Now we upgrade to the full model (SIE + shear) and use
# Search 1's posteriors as priors.
#
# Key pattern:
#   result_1.model.galaxies.lens.mass  → Gaussian priors from posterior
#   result_1.instance.galaxies.lens.mass → fixed to best-fit values
# ============================================================

# Lens: upgrade SIS → SIE, unfix centres, add shear
lens_2 = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=al.lp.Sersic,
    mass=al.mp.Isothermal,         # SIE (with ellipticity)
    shear=al.mp.ExternalShear,
)

# Transfer priors from Search 1 for shared parameters
# (centre, einstein_radius inherit Gaussian priors from posterior)
lens_2.mass.take_attributes(result_1.model.galaxies.lens.mass)
lens_2.bulge.take_attributes(result_1.model.galaxies.lens.bulge)

# Source: transfer priors from Search 1
source_2 = af.Model(
    al.Galaxy,
    redshift=1.0,
    bulge=al.lp.SersicCore,
)
source_2.bulge.take_attributes(result_1.model.galaxies.source.bulge)

model_2 = af.Collection(
    galaxies=af.Collection(lens=lens_2, source=source_2)
)

print(f"Search 2: {model_2.total_free_parameters} free parameters")
print("(But starting near the truth thanks to Search 1!)")

search_2 = af.Nautilus(
    path_prefix=Path("output") / "module_04" / "chaining",
    name="search_2_full",
    unique_tag=dataset_name,
    n_live=100,
)

result_2 = search_2.fit(model=model_2, analysis=analysis)

print(f"\nSearch 2 complete.")
print(result_2.info)

In [ ]:
# ============================================================
# COMPARE: SEARCH 1 vs. SEARCH 2
# ============================================================
# The chained result should be significantly better than
# Search 1 alone, with structured residuals from Search 1
# cleaned up in Search 2.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, result, title in zip(
    axes,
    [result_1, result_2],
    ["Search 1: SIS (simplified)", "Search 2: SIE + shear (full)"]
):
    residuals = result.max_log_likelihood_fit.residual_map.native
    im = ax.imshow(
        residuals,
        origin="lower",
        cmap="RdBu_r",
        vmin=-3, vmax=3,
    )
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, label="Normalized residuals")

plt.suptitle("Residual Improvement from Search Chaining", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## 4. The SLaM Pipeline: Architecture <a id="4-slam-architecture"></a>

### SLaM = Source, Light, and Mass

The SLaM pipeline is PyAutoLens's standardized search chaining strategy, designed
to robustly model galaxy-scale lenses with lens light. It has **four stages**:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   SOURCE LP      │ ──→│   SOURCE PIX     │ ──→│    LIGHT LP      │ ──→│   MASS TOTAL     │
│                  │     │                  │     │                  │     │                  │
│ Parametric source│     │ Pixelized source │     │ Refined lens     │     │ Complex mass     │
│ Simple SIE mass  │     │ Refined mass     │     │ light model      │     │ model (PowerLaw) │
│ Parametric light │     │ Fixed light      │     │ Fixed mass+source│     │ Fixed light      │
└─────────────────┘     └─────────────────┘     └─────────────────┘     └─────────────────┘
```

### Why This Order?

1. **Source first**: Getting the source right is critical because it determines
   how well we can deblend the lens light. A pixelized source is much more
   flexible than a parametric one.

2. **Light before mass**: Accurate lens light subtraction is needed before
   fitting complex mass models (whose residuals would be confused with
   lens light errors).

3. **Mass last**: With light and source locked in, the mass model search
   has a clean likelihood surface.

### What Each Stage Does

| Stage | Free | Fixed | Purpose |
|-------|------|-------|----------|
| **SOURCE LP** | lens light, mass (SIE), source (Sérsic) | — | Initial rough model |
| **SOURCE PIX** | source pixelization, mass | lens light | Flexible source reconstruction |
| **LIGHT LP** | lens light | mass, source | Refined lens light with accurate source |
| **MASS TOTAL** | mass (PowerLaw) | lens light, source | Final mass model |

---

## 5. SLaM Stage 1: SOURCE LP (Parametric Source) <a id="5-source-lp"></a>

### What It Does

The first search fits everything simultaneously with a **simple, parametric** model:
- Lens light: Sérsic (or multi-Gaussian expansion)
- Lens mass: SIE + external shear
- Source: Sérsic

This is similar to what we did in Module 03, but with lens light included.
The key output is a **rough but reasonable** model that initializes everything else.

In [ ]:
# ============================================================
# SLAM STAGE 1: SOURCE LP
# ============================================================
# The source_lp.run() function wraps all the model setup,
# search configuration, and fitting into a single call.
#
# Key parameters:
#   lens_bulge: light profile model for the lens
#   mass: mass profile model (default: Isothermal/SIE)
#   shear: external shear model
#   source_bulge: light profile model for the source
#   mass_centre: if provided, fixes the mass centre
# ============================================================

# Configure search settings
settings_search = af.SettingsSearch(
    path_prefix=Path("output") / "module_04" / "slam",
    unique_tag=dataset_name,
)

# Analysis object (reusable across SLaM stages)
analysis_source = al.AnalysisImaging(dataset=dataset)

# Run SOURCE LP
source_lp_result = source_lp.run(
    settings_search=settings_search,
    analysis=analysis_source,
    lens_bulge=af.Model(al.lp.Sersic),
    lens_disk=None,                        # No disk component
    mass=af.Model(al.mp.Isothermal),       # SIE
    shear=af.Model(al.mp.ExternalShear),
    source_bulge=af.Model(al.lp.Sersic),
    redshift_lens=0.5,
    redshift_source=1.0,
    mass_centre=(0.0, 0.0),                # Fix mass centre for stability
)

print("SOURCE LP complete.")
print(f"Best θ_E = {source_lp_result.instance.galaxies.lens.mass.einstein_radius:.3f}\"")

In [ ]:
# Visualize the SOURCE LP result
fit_plotter = aplt.FitImagingPlotter(fit=source_lp_result.max_log_likelihood_fit)
fit_plotter.subplot_fit()

---

## 6. SLaM Stage 2: SOURCE PIX (Pixelized Source) <a id="6-source-pix"></a>

### Theory: Why Pixelized Sources?

A Sérsic profile can't capture irregular source morphology (spiral arms, clumps,
merging galaxies). A **pixelized source** reconstructs the source brightness on a
mesh (Delaunay triangulation, Voronoi, or rectangular grid) in the source plane.

This is a linear inversion: given the mass model (which defines the ray-tracing),
the source pixel brightnesses are solved analytically via:

$$
\mathbf{s} = (\mathbf{F}^T \mathbf{C}^{-1} \mathbf{F} + \lambda \mathbf{H})^{-1} \mathbf{F}^T \mathbf{C}^{-1} \mathbf{d}
$$

where $\mathbf{F}$ is the lensing operator (ray-tracing + PSF), $\mathbf{C}$ is the
noise covariance, $\mathbf{H}$ is the regularization matrix, and $\lambda$ controls
smoothness (Suyu et al. 2006; Nightingale+18 Sec. 5).

### The Two Sub-Searches

SOURCE PIX runs two searches:
1. **run_1**: Initialize with a simple Overlay mesh (uniform grid in image plane)
2. **run_2**: Refine with an adaptive Hilbert mesh (concentrates pixels where the source is bright)

In [ ]:
# ============================================================
# SLAM STAGE 2a: SOURCE PIX — SEARCH 1 (Initialize)
# ============================================================
# Uses the SOURCE LP result to:
#   - Fix lens light to best-fit values
#   - Initialize mass priors from SOURCE LP posteriors
#   - Replace parametric source with pixelized source
#
# The AdaptImageMaker creates an "adapt image" that tells the
# pixelization where to concentrate resolution.
# ============================================================

analysis_pix = al.AnalysisImaging(
    dataset=dataset,
    adapt_image_maker=al.AdaptImageMaker(result=source_lp_result),
)

source_pix_result_1 = source_pix.run_1(
    settings_search=settings_search,
    analysis=analysis_pix,
    source_lp_result=source_lp_result,
    mesh_init=al.mesh.Delaunay,
)

print("SOURCE PIX (search 1) complete.")

In [ ]:
# ============================================================
# SLAM STAGE 2b: SOURCE PIX — SEARCH 2 (Refine)
# ============================================================
# Refines the pixelization using a Hilbert curve image mesh,
# which adaptively places more pixels where the source is
# brighter. This gives much better source reconstructions.
# ============================================================

analysis_pix_2 = al.AnalysisImaging(
    dataset=dataset,
    adapt_image_maker=al.AdaptImageMaker(result=source_pix_result_1),
    settings_inversion=al.SettingsInversion(
        image_mesh_min_mesh_pixels_per_pixel=3,
        image_mesh_adapt_background_percent_threshold=0.1,
    ),
)

source_pix_result_2 = source_pix.run_2(
    settings_search=settings_search,
    analysis=analysis_pix_2,
    source_lp_result=source_lp_result,
    source_pix_result_1=source_pix_result_1,
    image_mesh=al.image_mesh.Hilbert,
    mesh=al.mesh.Delaunay,
    regularization=al.reg.AdaptiveBrightnessSplit,
)

print("SOURCE PIX (search 2) complete.")

In [ ]:
# Visualize the pixelized source reconstruction
fit_plotter = aplt.FitImagingPlotter(fit=source_pix_result_2.max_log_likelihood_fit)
fit_plotter.subplot_fit()

---

## 7. SLaM Stage 3: LIGHT LP (Lens Light) <a id="7-light-lp"></a>

### Purpose

With an accurate source model (pixelized), we can now refit the lens light.
The lens mass and source are **fixed** while the lens light parameters are **free**.

This stage often significantly improves the lens light model because the source
subtraction is now much more accurate than in SOURCE LP.

In [ ]:
# ============================================================
# SLAM STAGE 3: LIGHT LP
# ============================================================
# Refits lens light with mass and source locked.
# This deblends the lens light cleanly because the pixelized
# source accurately captures the arc emission.
# ============================================================

analysis_light = al.AnalysisImaging(
    dataset=dataset,
    adapt_image_maker=al.AdaptImageMaker(result=source_pix_result_1),
)

light_result = light_lp.run(
    settings_search=settings_search,
    analysis=analysis_light,
    source_result_for_lens=source_pix_result_1,
    source_result_for_source=source_pix_result_2,
    lens_bulge=af.Model(al.lp.Sersic),
)

print("LIGHT LP complete.")

---

## 8. SLaM Stage 4: MASS TOTAL (Final Mass Model) <a id="8-mass-total"></a>

### Purpose

The final stage fits the mass model with everything else locked in. We can now
upgrade from the simple SIE to a more flexible model:

- **PowerLaw**: $\kappa \propto \theta^{1-\gamma'}$ with free slope $\gamma'$
  (SIE is the special case $\gamma' = 2$)
- **Composite**: separate stellar (Sérsic) + dark matter (NFW) mass components
- **Multipole**: higher-order angular structure

In [ ]:
# ============================================================
# SLAM STAGE 4: MASS TOTAL
# ============================================================
# Fit the final mass model. Here we use PowerLaw, which
# generalizes SIE with a free density slope.
#
# This is the publishable result: the best mass model with
# accurately deblended light and flexible source.
# ============================================================

analysis_mass = al.AnalysisImaging(
    dataset=dataset,
    adapt_image_maker=al.AdaptImageMaker(result=source_pix_result_1),
)

mass_result = mass_total.run(
    settings_search=settings_search,
    analysis=analysis_mass,
    source_result_for_lens=source_pix_result_1,
    source_result_for_source=source_pix_result_2,
    light_result=light_result,
    mass=af.Model(al.mp.Isothermal),   # Use Isothermal for speed; PowerLaw for science
)

print("MASS TOTAL complete.")
print(mass_result.info)

In [ ]:
# ============================================================
# FINAL RESULT VISUALIZATION
# ============================================================
fit_plotter = aplt.FitImagingPlotter(fit=mass_result.max_log_likelihood_fit)
fit_plotter.subplot_fit()

---

## 9. Putting It All Together: Full SLaM Run <a id="9-full-slam"></a>

The complete SLaM pipeline for a new lens system is:

```python
# 1. SOURCE LP: rough parametric model
source_lp_result = source_lp.run(...)

# 2. SOURCE PIX: flexible pixelized source
source_pix_result_1 = source_pix.run_1(..., source_lp_result=source_lp_result)
source_pix_result_2 = source_pix.run_2(..., source_lp_result=source_lp_result,
                                             source_pix_result_1=source_pix_result_1)

# 3. LIGHT LP: refined lens light
light_result = light_lp.run(..., source_result_for_lens=source_pix_result_1,
                                 source_result_for_source=source_pix_result_2)

# 4. MASS TOTAL: final mass model
mass_result = mass_total.run(..., source_result_for_lens=source_pix_result_1,
                                  source_result_for_source=source_pix_result_2,
                                  light_result=light_result)
```

This pattern is **general**: it works for any galaxy-scale lens with the same 5 searches.
The only things that change between targets are:
- The dataset (FITS files)
- The mask radius
- The redshifts
- (Optionally) the mass model complexity

---

## 10. Exercises <a id="10-exercises"></a>

### Exercise 1: Three-Search Chain

Design a three-search chain for a lens WITHOUT lens light:
1. Search 1: SIS mass + exponential source
2. Search 2: SIE + shear mass + Sérsic source (priors from Search 1)
3. Search 3: PowerLaw + shear mass + Sérsic source (priors from Search 2)

Does the PowerLaw slope $\gamma'$ differ significantly from 2 (the SIE value)?

### Exercise 2: Instance vs. Model Passing

In Search 2 of a chain, compare:
- **Instance passing**: `lens_light = result_1.instance.galaxies.lens.bulge`
- **Model passing**: `lens_light = result_1.model.galaxies.lens.bulge`

Which gives better final results? Why? (Think about whether the lens light
from Search 1 is good enough to fix.)

### Exercise 3: SLaM on a Different Dataset

Run the full SLaM pipeline on the `lens_sersic` dataset:
```python
dataset_path = Path("../../autolens_workspace_original/dataset/imaging/lens_sersic")
```
Compare the lens light model from LIGHT LP with the input Sérsic parameters.

### Exercise 4: Timing Comparison

Time the full SLaM pipeline vs. a single search with all parameters free.
How much faster is SLaM? (SLaM should be 5–10× faster for ~25-parameter models.)

---

## Summary

| Concept | Implementation | Why It Matters |
|---------|---------------|----------------|
| Search chaining | `result.model` / `result.instance` / `take_attributes()` | Breaks high-dimensional fits into tractable stages |
| SLaM: SOURCE LP | `source_lp.run()` | Initial rough model |
| SLaM: SOURCE PIX | `source_pix.run_1()`, `run_2()` | Flexible non-parametric source |
| SLaM: LIGHT LP | `light_lp.run()` | Clean lens light deblending |
| SLaM: MASS TOTAL | `mass_total.run()` | Final publishable mass model |

**Next module:** We'll dive deeper into the pixelized source reconstructions — how
the meshes, regularization, and inversions work under the hood.

---

*Learning to Autolens — Module 04*
*Rodrigo Córdova Rosado, Harvard CfA*
*Built with Claude Code*